# Pie Charts, Box Plots, Scatter Plots, và Bubble Plots

## Mục tiêu

Sau khi hoàn thành lab này, bạn có thể:

*   Tìm hiểu sâu hơn về các thư viện Matplotlib
*   Tạo ra các biểu đồ pie charts, box plots, scatter plots và bubble charts

# Chuẩn bị dữ liệu <a id="2"></a>


Import thư viện numpy và pandas

In [ ]:
import numpy as np  
import pandas as pd 

Load dữ liệu:

In [ ]:
df_can = pd.read_excel('datasets/canadian_immegration_data.xlsx',
                       sheet_name='Canada by Citizenship',
                       skiprows=range(20),
                       skipfooter=2
                      )

print('Finished load data!')

Xem top 5 dòng dữ liệu đầu tiên bằng hàm `head()`.

In [ ]:
df_can.head()

Làm sạch dữ liệu. Chúng ta cần phải thực hiện một số biến đối với tập dữ liệu gốc để giúp tạo trực quan hóa dễ dàng hơn. Tham khảo Lecture 8 bài tập 1 để hiểu hơn về các bước sau đây:

In [ ]:
# Xoá và loại bỏ các cột không cần thiết
df_can.drop(['AREA', 'REG', 'DEV', 'Type', 'Coverage'], axis=1, inplace=True)

# Đổi tên các cột để mang ý nghĩa hơn
df_can.rename(columns={'OdName':'Country', 'AreaName':'Continent','RegName':'Region'}, inplace=True)

# Biến đổi tên các cột thành định dạng chuỗi
df_can.columns = list(map(str, df_can.columns))

# Thiết lập cột country là cột chỉ mục - nhằm mục đích tìm kiếm dễ hơn với phương thức .loc
df_can.set_index('Country', inplace=True)

# Thêm cột total 
df_can['Total'] = df_can.sum(axis=1, numeric_only=True)

# Liệt kê các năm dữ liệu được sử dụng trong lab này
years = list(map(str, range(1980, 2014)))
print('data dimensions:', df_can.shape)

# Trực quan hoá dữ liệu bằng thư viện Matplotlib<a id="4"></a>


Import `Matplotlib`.


In [ ]:
%matplotlib inline

import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('ggplot')  

# Kiểm tra phiên bản hiện tại của thư viện Matplotlib
print('Matplotlib version: ', mpl.__version__) # >= 2.0.0

# Pie Charts <a id="6"></a>
`pie chart` là biểu đồ hình tròn hiển thị tỷ lệ số bằng cách chia hình tròn thành các lát theo tỷ lệ. Rất có thể bạn đã quen thuộc với biểu đồ hình tròn vì nó được sử dụng rộng rãi trong kinh doanh và truyền thông.

Chúng ta có thể tạo một biểu đồ hình tròn bằng cách truyền tham số `kind=pie`

Chúng ta hãy sử dụng biểu đồ hình tròn để khám phá tỷ lệ (phần trăm) số người nhập cư mới được nhóm theo các châu lục trong toàn bộ khoảng thời gian từ năm 1980 đến năm 2013.


**Bước 1:** Nhóm dữ liệu:

Chúng ta sẽ sư dụng phương thức *pandas* `groupby` để tóm tắt dữ liệu nhập cư bằng cột `Continent`. Quy trình chung của `groupby` bao gồm các bước sau:

1.  **Split:** Chia dữ liệu thành các nhóm dựa trên một số tiêu chí.
2.  **Apply:** Áp dụng một hàm tính toán cho từng nhóm một cách độc lập:
    .sum()
    .count()
    .mean()
    .std()
    .aggregate()
    .apply()
    .etc..
3.  **Combine:** Kết hợp các kết quả vào một dataframe.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%203/images/Mod3Fig4SplitApplyCombine.png" height=400 align="center">


In [ ]:
# group countries bằng cột continents and áp dụng hàm sum() 
df_continents = df_can.groupby('Continent', axis=0).sum()

df_continents.head()

**Bước 2:** Vẽ biểu đồ. Học viên cần phải truyền tham số `kind = 'pie'` vào hàm plot, cùng với một vài tham số khác:

*   `autopct` - là một chuỗi hoặc hàm dùng để gắn nhãn các lát tỷ lệ của biểu đồ với giá trị của chúng. Nhãn sẽ được xuất hiện trong từng lát tỷ lệ của biểu đồ. Nếu nhãn có định dạng là chuỗi, thì sẽ có format kiểu: `fmt%pct`.
*   `startangle` - xoay phần đầu của biểu đồ hình tròn theo góc độ ngược chiều kim đồng hồ so với trục x.
*   `shadow` - Tạo hiện tượng đổ bóng cho biểu đồ (để tạo cảm giác 3D).


In [ ]:
# autopct create %, start angle represent starting point
df_continents['Total'].plot(kind='pie',
                            figsize=(5, 6),
                            autopct='%1.1f%%', # add in percentages
                            startangle=90,     # start angle 90° (Africa)
                            shadow=True,       # add shadow      
                            )

plt.title('Tỷ lệ nhập cư vào Canada theo lục địa [1980 - 2013]')
plt.axis('equal') # Sets the pie chart to look like a circle.

plt.show()

Hình ảnh trên không rõ ràng lắm, trong một số trường hợp, số và nhãn có thể chồng chéo lên nhau. Chúng ta cùng thực hiện một số sửa đổi để cải thiện chất lượng của biểu đồ:

*   Xóa nhãn trên biểu đồ hình tròn bằng cách chuyển chúng vào `legend` và thêm nó dưới dạng chú thích riêng biệt bằng cách sử dụng `plt.legend()`.
*   Đẩy tỷ lệ phần trăm ra ngay bên ngoài biểu đồ hình tròn bằng cách sử dụng tham số `pctdistance`.
*   Truyền vào một tập hợp màu tùy chỉnh cho các châu lục bằng cách truyền vào tham số `colors`.
*   **Explode** Nếu bạn muốn ba lục địa thấp nhất (Châu Phi, Bắc Mỹ, Châu Mỹ Latinh và Caribe) là điểm nhấn chính trong biểu đồ hình tròn, bạn có thể truyền tham số `explode` như đoạn code sau đây.


In [ ]:
colors_list = ['gold', 'yellowgreen', 'lightcoral', 'lightskyblue', 'lightgreen', 'pink']
explode_list = [0.1, 0, 0, 0, 0.1, 0.1] # ratio for each continent with which to offset each wedge.

df_continents['Total'].plot(kind='pie',
                            figsize=(15, 6),
                            autopct='%1.1f%%', 
                            startangle=90,    
                            shadow=True,       
                            labels=None,         
                            pctdistance=1.12,     
                            colors=colors_list, 
                            explode=explode_list 
                            )

# scale the title up by 12% to match pctdistance
plt.title('Tỷ lệ nhập cư vào Canada theo lục địa [1980 - 2013]', y=1.12) 

plt.axis('equal') 

# add legend
plt.legend(labels=df_continents.index, loc='upper left') 

plt.show()

### Bài tập 1: 
Sử dụng biểu đồ hình tròn, hãy khám phá tỷ lệ (phần trăm) số người nhập cư mới được nhóm theo các châu lục trong năm 2013.

**Lưu ý**: Bạn có thể cần thử nghiệm các giá trị `explore` để cải thiện biểu đồ và tránh chồng chéo giữa các tỷ lệ.

In [ ]:
### Code của bạn ở đây

# Box Plots <a id="8"></a>

`box plot` là một biểu đồ hình hộp biểu diễn thống kê *phân phối* của dữ liệu thông qua năm yếu tố chính:

*   **Minimum:** Số nhỏ nhất trong tập dữ liệu không bao gồm các giá trị ngoại lệ.
*   **First quartile:** Số ở giữa giữa `Minimum` và `Median`.
*   **Second quartile (Median):** Số giữa của tập dữ liệu đã được sắp xếp.
*   **Third quartile:** Số ở giữa giữa `median` và `maximum`.
*   **Maximum:** Số lớn nhất trong tập dữ liệu không bao gồm các giá trị ngoại lệ.


<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%203/images/boxplot_complete.png" width=440, align="center">


Để vẽ một biểu đồ `boxplot`, chúng ta cần truyền tham số `kind=box` trong phương thức `plot`: 

Hãy vẽ đồ thị hình hộp về người nhập cư đến từ Việt Nam trong khoảng thời gian 1980 - 2013.

**Bước 1:** Lấy tập hợp con của tập dữ liệu. Mặc dù chúng ta chỉ lấy dữ liệu của một quốc gia, chúng ta cần lấy dữ liệu đó dưới dạng dataframe.

In [ ]:
# Lấy thông tin số lượng nhập cư đến từ Việt Nam
df_vn = df_can.loc[['Viet Nam'], years].transpose()
df_vn.head()

**Bước 2:** Vẻ biểu đồ bằng cách truyền tham số `kind='box'`.


In [ ]:
df_vn.plot(kind='box', figsize=(8, 6))

plt.title('Box plot mô tả số người nhập cư từ Việt Nam từ 1980 - 2013')
plt.ylabel('Số người')

plt.show()

Chúng ta có thể ngay lập tức đưa ra một số nhận xét quan trọng từ biểu đồ trên:

1.  Số lượng người nhập cư tối thiểu là khoảng 1200 (tối thiểu), số lượng tối đa là khoảng 5400 (tối đa) và trung vị là khoảng 2200 (Median) `Lưu ý: Không bao gồm dữ liệu ngoại lệ`.
2.  25% số năm trong giai đoạn 1980 - 2013 có số lượng người nhập cư hàng năm nhỏ hơn hoặc bằng \~1700 (`First quartile`).
3.  75% số năm trong giai đoạn 1980 - 2013 có số lượng người nhập cư hàng năm nhỏ hơn hoặc bằng \~3300 (`Third quartile`).

Chúng ta có thể xem các con số thực tế bằng cách gọi phương thức `describe()` trên dataframe.

In [ ]:
df_vn.describe()

### Bài tập 2

Một trong những lợi ích chính của box plot là so sánh sự phân bố của nhiều tập dữ liệu. Tại lab trước, chúng ta quan sát thấy Trung Quốc và Ấn Độ có xu hướng nhập cư rất giống nhau. Hãy phân tích sâu hơn về hai quốc gia này bằng cách sử dụng biểu đồ hình hộp.

**Bài tập:** 
So sánh sự phân bố số lượng người nhập cư mới từ Ấn Độ và Trung Quốc trong giai đoạn 1980 - 2013.

**Bước 1**: Lấy tập dữ liệu cho Trung Quốc và Ấn Độ và đặt tên dataframe là **df_CI**.

In [ ]:
### Code của bạn ở đây

Phân tích thống kê mô tả bằng hàm `.describe()`

In [ ]:
### Code của bạn ở đây

**Bước 2**: Vẽ biểu đồ

In [ ]:
### Code của bạn ở đây

Chúng ta có thể quan sát thấy rằng, mặc dù cả hai quốc gia đều có dân số nhập cư trung bình như nhau (\~20.000), nhưng phạm vi dân số nhập cư của Trung Quốc lại trải rộng hơn so với Ấn Độ. 

Số lượng người nhập cư tối đa đến từ Ấn Độ trong bất kỳ năm nào (36.210) thấp hơn khoảng 15% so với số lượng người nhập cư tối đa đến từ Trung Quốc (42.584).

Nếu muốn tạo các box plot theo chiều ngang, bạn có thể truyền tham số `vert=False` trong hàm **plot**. Bạn cũng có thể chỉ định một màu khác trong trường hợp bạn không phải là người thích màu đỏ mặc định.

In [ ]:
# horizontal box plots
df_CI.plot(kind='box', figsize=(10, 7), color='blue')

plt.title('Box plot mô tả số người nhập cư từ Trung Quốc và Ấn Độ (1980 - 2013)')
plt.xlabel('Số lượng')

plt.show()

**Subplots**


Thông thường, chúng ta có thể muốn vẽ nhiều biểu đồ trong cùng một hình. Ví dụ: chúng ta có thể muốn thực hiện so sánh song song biểu đồ hình hộp với biểu đồ đường thẳng về sự nhập cư của Trung Quốc và Ấn Độ.

Để trực quan hóa nhiều biểu đồ cùng nhau, chúng ta có thể tạo một **`figure`** (khung vẽ tổng thể) và chia nó thành **`subplots**, mỗi ô chứa một biểu đồ. Với **subplots**, chúng ta thường làm việc với **artist layer** thay vì **scripting layer**.

Các syntax cơ bản: <br>

```python
    fig = plt.figure() # create figure
    ax = fig.add_subplot(nrows, ncols, plot_number) # create subplots
```

Trong đó

*   `nrows` và `ncols` được sử dụng để phân chia hình thành các trục phụ (`nrows` \* `ncols`),
*   `plot_number`  được sử dụng để xác định ô con cụ thể mà hàm này sẽ tạo. `plot_number` bắt đầu từ 1, tăng dần qua các hàng từ trái qua phải và có tối đa là `nrows` \* `ncols` như hiển thị bên dưới.

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%203/images/Mod3Fig5Subplots_V2.png" width=500 align="center">


Sau đó, chúng ta có thể chỉ định ô phụ nào sẽ đặt mỗi ô bằng cách chuyển tham số `ax` trong phương thức `plot()` như sau:

In [ ]:
fig = plt.figure() # create figure

ax0 = fig.add_subplot(1, 2, 1) # add subplot 1 (1 row, 2 columns, first plot)
ax1 = fig.add_subplot(1, 2, 2) # add subplot 2 (1 row, 2 columns, second plot). See tip below**

# Subplot 1: Box plot
df_CI.plot(kind='box', color='blue', vert=False, figsize=(20, 6), ax=ax0) # add to subplot 1
ax0.set_title('Box Plots of Immigrants from China and India (1980 - 2013)')
ax0.set_xlabel('Number of Immigrants')
ax0.set_ylabel('Countries')

# Subplot 2: Line plot
df_CI.plot(kind='line', figsize=(20, 6), ax=ax1) # add to subplot 2
ax1.set_title ('Line Plots of Immigrants from China and India (1980 - 2013)')
ax1.set_ylabel('Number of Immigrants')
ax1.set_xlabel('Years')

plt.show()

**Một vài mẹo khi sử dụng subplot**

Trong trường hợp các tham số `nrows`, `ncols`, và `plot_number` nhỏ hơn 10, có một sự tiện lợi là có thể đưa ra một số có 3 chữ số thay thế, khi mà chữ số hàng trăm đại diện cho `nrows`, chữ số hàng chục đại diện cho `ncols` và hàng đơn vị đại diện cho `plot_number`. Ví dụ:

```python
   subplot(211) == subplot(2, 1, 1) 
```


### Bài tập 3:

Trước đây chúng ta đã xác định 15 quốc gia có số lượng nhập cư vào canada hàng đầu dựa trên số liệu tổng số người nhập cư từ năm 1980 - 2013.

Tạo một biểu đồ hình hộp để trực quan hóa sự phân bổ của 15 quốc gia hàng đầu (dựa trên tổng số người nhập cư) được nhóm theo *thập kỷ* `thập niên 1980`, `thập niên 1990` và `thập niên 2000`.

**Bước 1**: Lấy thông tin 15 quốc gia hàng đầu dựa trên Tổng dân số nhập cư (cột Total). Đặt tên cho dataframe **df_top15**.

In [ ]:
### type your answer here

# df_top15 = ...


**Bước 2**: Tạo một dataframe mới chứa thông tin tổng hợp cho mỗi thập kỷ. Bằng cách:

1.  Tạo danh sách tất cả các năm trong thập kỷ 80's, 90's, và 2000's.
2.  Cắt khung dữ liệu ban đầu df_can để tạo một chuỗi cho mỗi thập kỷ và tính tổng tất cả các năm cho mỗi quốc gia
3.  Hợp nhất ba chuỗi vào một dataframe mới. Gọi dataframe mới là **new_df**.


In [ ]:
### Code của bạn ở đây


# Cắt khung dữ liệu ban đầu df_can để tạo một chuỗi cho mỗi thập kỷ và tính tổng tất cả các năm cho mỗi quốc gia


# Hợp nhất ba chuỗi vào một dataframe mới. Gọi dataframe mới là new_df.

# display dataframe


Sử dụng phương thức `describe()` để xem thông tin thống kê mô tả:


In [ ]:
### Code của bạn ở đây


**Bước 3**: Vẽ biểu đồ Box Plot


In [ ]:
### Code của bạn ở đây

new_df.plot(kind='box', figsize=(10, 6))

plt.title('Nhập cư từ 15 quốc gia hàng đầu trong thập niên 80, 90 và 2000')

plt.show()

Lưu ý biểu đồ hộp khác với bảng tóm tắt được như thế nào. Biểu đồ hộp quét dữ liệu và xác định các giá trị ngoại lai (outliers) . Để trở thành một ngoại lai, giá trị dữ liệu phải là:<br>


*   Lớn hơn Q3 ít nhất bằng 1.5 lần (IQR)
*   Nhỏ hơn Q1 ít nhất bằng 1.5 lần (IQR)

Ví dụ cho thập niên 2000s: <br>

*   Q1 (25%) = 36,101.5 <br>
*   Q3 (75%) = 105,505.5 <br>
*   IQR = Q3 - Q1 = 69,404 <br>

Sử dụng định nghĩa về ngoại lệ, bất kỳ giá trị nào lớn hơn Q3 1,5 lần IQR sẽ được gắn cờ là ngoại lệ.

Outlier > 105,505.5 + (1.5 \* 69,404) <br>
Outlier > 209,611.5


In [ ]:
# Kiểm tra xem có bao nhiêu mục nằm trên ngưỡng ngoại lệ
new_df=new_df.reset_index()
new_df[new_df['2000s']> 209611.5]

<!-- The correct answer is:
new_df[new_df['2000s']> 209611.5]
-->


Trung Quốc và Ấn Độ đều được coi là những trường hợp ngoại lai vì dân số của họ trong thập kỷ này vượt quá 209.611,5.


Để tìm hiểu và cá nhân hoá thêm về biểu đồ box plot, bạn có thể tham khảo [Matplotlib documentation](http://matplotlib.org/api/pyplot_api.html?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork20297740-2021-01-01#matplotlib.pyplot.boxplot) 


# Scatter Plots <a id="10"></a>


`scatter plot` (2D) là một phương pháp hữu ích để so sánh các biến với nhau. Biểu đồ `Scatter` trông giống như `line plots` ở chỗ chúng đều ánh xạ các biến độc lập và biến phụ thuộc trên biểu đồ 2D. Mặc dù các điểm dữ liệu được kết nối với nhau bằng một đường trong biểu đồ `line plots` nhưng chúng không được kết nối trong biểu đồ `scatter plot`. Dữ liệu trong biểu đồ phân tán được coi là thể hiện một xu hướng. Với những phân tích sâu hơn bằng cách sử dụng các công cụ như hồi quy, chúng ta có thể tính toán mối quan hệ này một cách toán học và sử dụng nó để dự đoán các xu hướng bên ngoài tập dữ liệu.


Hãy bắt đầu bằng cách khám phá những điều sau:

**Bước 1** Sử dụng `Scatter Plots`, chúng ta hãy hình dung xu hướng tổng số người nhập cư vào Canada (tất cả các quốc gia cộng lại) trong những năm 1980 - 2013.

In [ ]:
# Chúng ta có thể sử dụng phương thức sum() để lấy tổng dân số mỗi năm
df_tot = pd.DataFrame(df_can[years].sum(axis=0))

# Thay đổi năm thành kiểu int (hữu ích cho việc hồi quy sau này)
df_tot.index = map(int, df_tot.index)

# reset the index
df_tot.reset_index(inplace = True)

# Đổi tên cột
df_tot.columns = ['year', 'total']

# Kết quả cuối cùng
df_tot.head()

**Bước 2**  Vẽ biểu đồ. Trong `Matplotlib`, chúng ta có thể vẻ biểu đồ `scatter` plot bằng cách truyền tham số `kind='scatter'` 
Chúng ta cũng sẽ cần chuyển vào từ khóa `x` và `y` để chỉ định các cột nằm trên trục x và trục y.

In [ ]:
df_tot.plot(kind='scatter', x='year', y='total', figsize=(10, 6), color='darkblue')
plt.title('Tổng số người nhập cư vào Canada từ 1980 - 2013')
plt.xlabel('Năm')
plt.ylabel('Số lượng')
plt.show()

Lưu ý rằng biểu đồ phân tán không kết nối các điểm dữ liệu với nhau. Chúng ta có thể quan sát rõ ràng xu hướng tăng lên trong dữ liệu: theo năm tháng, tổng số người nhập cư tăng lên. Chúng ta có thể phân tích một cách toán học xu hướng đi lên này bằng cách sử dụng đường hồi quy (đường phù hợp nhất).

<strong>Nâng cao:</strong> Vì vậy, hãy thử vẽ một đường tuyến tính phù hợp nhất và sử dụng nó để dự đoán số lượng người nhập cư vào năm 2015.

**Bước 1**: Tìm phương trình đường thẳng phù hợp nhất. Chúng ta sẽ sử dụng phương thức `polyfit()` của **Numpy** bằng cách truyền vào đoạn mã sau:

*   `x`: Điểm dữ liệu trung hoành (year).
*   `y`: Điểm dữ liệu trung tung (total).
*   `deg`: Mức độ phù hợp của đa thức. Ví dụ: 1 = linear, 2 = quadratic


In [ ]:
x = df_tot['year']      # year on x-axis
y = df_tot['total']     # total on y-axis
fit = np.polyfit(x, y, deg=1)

fit

Đầu ra là một mảng có các hệ số đa thức. Vì chúng ta đang vẽ biểu đồ hồi quy tuyến tính `y= a * x + b` nên đầu ra của chúng ta có 2 phần tử `[5.56709228e+03, -1.09261952e+07]` với độ dốc ở vị trí 0 và intercept ở vị trí 1.

**Bước 2**: Vẽ đường hồi quy trên `scatter plot`.


In [ ]:
df_tot.plot(kind='scatter', x='year', y='total', figsize=(10, 6), color='darkblue')

plt.title('Tổng số người nhập cư vào Canada từ 1980 - 2013')
plt.xlabel('Năm')
plt.ylabel('Số lượng')

# plot line of best fit
plt.plot(x, fit[0] * x + fit[1], color='red') # Với x là giá trị năm
plt.annotate('y={0:.0f} x + {1:.0f}'.format(fit[0], fit[1]), xy=(2000, 150000))
plt.show()

# print out the line of best fit
'Số lượng người nhập cư = {0:.0f} * Year + {1:.0f}'.format(fit[0], fit[1]) 

Sử dụng phương trình phù hợp nhất, chúng ta có thể ước tính số lượng người nhập cư vào năm 2015:

```python
No. Immigrants = 5567 * Year - 10926195
No. Immigrants = 5567 * 2015 - 10926195
No. Immigrants = 291,310
```

### Bài tập 4
Tạo một biểu đồ phân tán về tổng số người nhập cư từ Đan Mạch, Na Uy và Thụy Điển đến Canada từ năm 1980 đến năm 2013?

**Bước 1**: Thu thập dữ liệu:

1. Tạo dataframe chỉ bao gồm các thông tin từ Đan Mạch, Na Uy và Thụy Điển. Đặt tên là **df_countries**.
2. Tổng hợp số lượng người nhập cư ở cả ba quốc gia mỗi năm và chuyển kết quả thành một dataframe. Đặt tên cho dataframe mới này là **df_total**.
3. Đặt lại chỉ mục (reset_index).
4. Đổi tên các cột thành **year** và **total**.
5. Hiển thị dataframe kết quả.

In [ ]:
### Code của bạn ở đây

# Tạo dataframe chỉ bao gồm các thông tin từ Đan Mạch, Na Uy và Thụy Điển. Đặt tên là df_countries.

# Tổng hợp số lượng người nhập cư ở cả ba quốc gia mỗi năm và chuyển kết quả thành một dataframe. Đặt tên cho dataframe mới này là df_total.

# Đặt lại chỉ mục (reset_index).

# Đổi tên các cột thành year và total.

# Thay đổi kiểu dữ liệu cột year

# Hiển thị dataframe kết quả.


**Bước 2**: Tạo biểu đồ phân tán với cột total và year trong **df_total**.

In [ ]:
### Code của bạn ở đây

# add title and label to axes

# show plot


# Bubble Plots <a id="12"></a>

`bubble plot`  là một biến thể của `scatter plot` hiển thị ba chiều dữ liệu (x, y, z). Các điểm dữ liệu được thay thế bằng bong bóng và kích thước của bong bóng được xác định bởi biến thứ ba `z`, còn được gọi là trọng số. Trong `maplotlib`, chúng ta có thể chuyển một mảng hoặc tham số vô hướng sang tham số `s` thành `plot()`, tham số này chứa trọng số của từng điểm.

**Hãy bắt đầu bằng việc phân tích ảnh hưởng của cuộc đại suy thoái ở Argentina**.

Argentina đã trải qua một cuộc đại suy thoái từ năm 1998 đến năm 2002, gây ra tình trạng thất nghiệp lan rộng, bạo loạn, sự sụp đổ của chính phủ và vỡ nợ nước ngoài. Về thu nhập, hơn 50% người dân Argentina thuộc diện nghèo bởi vì cuộc khủng hoảng năm 2002.

Hãy phân tích ảnh hưởng của cuộc khủng hoảng này và so sánh làn sóng nhập cư của Argentina với nước láng giềng Brazil. Hãy thực hiện điều đó bằng cách sử dụng `bubble plot` về nhập cư từ Brazil và Argentina trong những năm 1980 - 2013. Chúng ta sẽ đặt trọng số cho bong bóng là giá trị *normalized* của dân số cho mỗi năm.

**Bước 1**: Lấy dữ liệu cho Brazil và Argentina. Giống như trong ví dụ trước, chúng ta sẽ chuyển đổi cột `year` thành kiểu int và đưa nó vào dataframe:

In [ ]:
# Biến đổi dataframe
df_can_t = df_can[years].transpose()

# Thay đổi kiểu dữ liệu của chỉ mục Years (the index) sang integer
df_can_t.index = map(int, df_can_t.index)

# Thay đổi tên của chỉ mục
df_can_t.index.name = 'Year'

# Đặt lại chỉ mục (reset_index), chuyển chỉ mục year thành một column.

df_can_t.reset_index(inplace=True)

# Kiểm tra lại dataframe
df_can_t.head()

**Bước 2**: Tạo trọng số chuẩn hóa.

Có nhiều phương pháp chuẩn hóa trong thống kê, mỗi phương pháp có cách sử dụng riêng. Trong trường hợp này, chúng ta sẽ sử dụng [feature scaling](https://en.wikipedia.org/wiki/Feature_scaling?utm_medium=Exinfluencer&utm_source=Exinfluencer&utm_content=000026UJ&utm_term=10006555&utm_id=NA-SkillsNetwork-Channel-SkillsNetworkCoursesIBMDeveloperSkillsNetworkDV0101ENSkillsNetwork20297740-2021-01-01) để đưa tất cả các giá trị vào phạm vi \[0, 1] với công thức chung là:

<img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DV0101EN-SkillsNetwork/labs/Module%203/images/Mod3Fig3FeatureScaling.png" align="center">

trong đó $X$ là giá trị ban đầu, $X'$ là giá trị chuẩn hóa tương ứng. Công thức đặt giá trị tối đa trong tập dữ liệu thành 1 và đặt giá trị tối thiểu thành 0. Các điểm dữ liệu còn lại được chia tỷ lệ thành giá trị trong khoảng 0-1 tương ứng.


In [ ]:
# Chuẩn hoá dữ liệu Brazil
norm_brazil = (df_can_t['Brazil'] - df_can_t['Brazil'].min()) / (df_can_t['Brazil'].max() - df_can_t['Brazil'].min())

# Chuẩn hoá dữ liệu Argentina
norm_argentina = (df_can_t['Argentina'] - df_can_t['Argentina'].min()) / (df_can_t['Argentina'].max() - df_can_t['Argentina'].min())

**Bước 3**: Vẽ biểu đồ dữ liệu.

* Để vẽ hai biểu đồ phân tán khác nhau trong một biểu đồ, chúng ta có thể gộp các trục này vào biểu đồ kia bằng cách chuyển nó qua tham số `ax`.
  
* Chúng ta cũng sẽ chuyển trọng số bằng tham số `s`. Cho rằng các trọng số được chuẩn hóa nằm trong khoảng 0-1, chúng sẽ không hiển thị trên biểu đồ. Vì vậy, chúng ta sẽ:
    *   nhân trọng số với 2000 để tăng tỷ lệ trên biểu đồ 
    *   thêm 10 để bù cho giá trị tối thiểu


In [ ]:
# Brazil
ax0 = df_can_t.plot(kind='scatter',
                    x='Year',
                    y='Brazil',
                    figsize=(14, 8),
                    alpha=0.5,  # transparency
                    color='green',
                    s=norm_brazil * 2000 + 10,  # pass in weights 
                    xlim=(1975, 2015)
                    )

# Argentina
ax1 = df_can_t.plot(kind='scatter',
                    x='Year',
                    y='Argentina',
                    alpha=0.5,
                    color="blue",
                    s=norm_argentina * 2000 + 10,
                    ax=ax0
                    )

ax0.set_ylabel('Số lượng')
ax0.set_title('Nhập cư từ Brazil và Argentina từ 1980 đến 2013')
ax0.legend(['Brazil', 'Argentina'], loc='upper left', fontsize='x-large')

Kích thước của bong bóng tương ứng với quy mô dân số nhập cư trong năm đó, so với dữ liệu giai đoạn 1980 - 2013. Bong bóng càng lớn thì càng có nhiều người nhập cư vào năm đó.

Từ biểu đồ trên, chúng ta có thể thấy lượng nhập cư từ Argentina tăng tương ứng trong cuộc đại suy thoái 1998 - 2002. Chúng ta cũng có thể quan sát thấy mức tăng đột biến tương tự vào khoảng năm 1985 đến năm 1993. Trên thực tế, Argentina đã trải qua một cuộc đại suy thoái từ năm 1974 đến năm 1990, ngay trước khi bắt đầu cuộc đại suy thoái 1998 - 2002.

Một lưu ý tương tự là Brazil đã hứng chịu *Hiệu ứng Samba* khi đồng Real (tiền tệ) của Brazil giảm gần 35% vào năm 1999. Người ta lo ngại về một cuộc khủng hoảng tài chính ở Nam Mỹ vì nhiều nước Nam Mỹ phụ thuộc nhiều vào xuất khẩu công nghiệp từ Brazil. Chính phủ Brazil sau đó đã áp dụng một chương trình thắt lưng buộc bụng và nền kinh tế dần dần phục hồi trong những năm qua, đỉnh điểm là sự bùng nổ vào năm 2010. Dữ liệu nhập cư phản ánh những sự kiện này.


### Bài tập 5
Tạo biểu đồ Bubble Plots về hoạt động nhập cư từ Trung Quốc và Ấn Độ để trực quan hóa bất kỳ sự khác biệt nào theo thời gian từ năm 1980 đến năm 2013. Bạn có thể sử dụng **df_can_t** mà chúng ta đã xác định và sử dụng trong ví dụ trước.

**Bước 1**: Tạo trọng số chuẩn hóa.

In [ ]:
### Code của bạn ở đây 

**Bước 2**: Vẽ biểu đồ dữ liệu.

In [ ]:
### Code của bạn ở đây